# Goal Inference

This example shows how to evaluate a a `genlm.control` model on the Planetarium Blocksworld goal-inference task.

* **Task**: Given a natural-language description, generate the PDDL goal S-expression that completes (:goal (and …)) for a provided problem.

* **Data**: [Planetarium] (Zuo et al., 2024) Blocksworld subset, filtered to goals of the form (:goal (and …)) and small instances (e.g., < 10 objects).

## Setup

First, install the dependencies for this domain. In the root directory, run:    

```bash
pip install -e .[goal_inference]
```

### Install Fast Downward and VAL

Download the Fast-Downward planner, and the VAL plan validator according to the instructions in under https://github.com/BatsResearch/planetarium (Linux only):

Fast-Downward via Apptainer
```bash
apptainer pull fast-downward.sif docker://aibasel/downward:latest
```
VAL download link might not work, follow instructions to download binary at: https://github.com/KCL-Planning/VAL
```bash
mkdir tmp
curl -o tmp/VAL.zip https://dev.azure.com/schlumberger/4e6bcb11-cd68-40fe-98a2-e3777bfec0a6/_apis/build/builds/77/artifacts?artifactName=linux64\&api-version=7.1\&%24format=zip
unzip tmp/VAL.zip -d tmp/
tar -xzvf tmp/linux64/*.tar.gz -C tmp/ --strip-components=1
```
clean up
```bash
rm -rf tmp
```
Make sure to add fast-downward.sif and VAL to your PATH or make aliases.

## Usage 

### Initialize the dataset and evaluator

In [ ]:
from genlm.eval.domains.goal_inference import (
    GoalInferenceDataset,
    GoalInferenceEvaluator,
)

In [ ]:
dataset = GoalInferenceDataset.from_hf_planetarium(
    split="train", subset="default", max_objects=2, domains=["blocksworld"]
)

evaluator = GoalInferenceEvaluator()

### Define a model adaptor

A model adaptor is an async callable that takes a `PatternMatchingInstance` and returns a `ModelOutput`. Here we'll use a `genlm.control.PromptedLLM` constrained to PDDL goals (via the `GoalInferenceVALPotential` potential).

In [ ]:
from genlm.control import PromptedLLM, AWRS
from genlm.eval import ModelOutput, ModelResponse
from genlm.eval.domains.goal_inference import (
    goal_default_prompt_formatter,
    GoalInferenceVALPotential,
    get_domain_text
)

# Load an LLM
LLM = PromptedLLM.from_name("gpt2", eos_tokens=[b"\n", b"\n\n"])


async def model(instance, output_dir, replicate):
    # Set the prompt for the LLM.
    LLM.prompt_ids = goal_default_prompt_formatter(
        LLM.model.tokenizer, instance, use_chat_format=False
    )

    domain_text = get_domain_text("blocksworld")
    potential = GoalInferenceVALPotential(
        domain_pddl_text=domain_text,
        problem_pddl_text=instance.problem_text,
        fast_downward_cmd="./fast-downward.sif",
        val_cmd="Validate",
        cache_root=".cache",
        verbosity=0,
    ).coerce(LLM, f=b"".join)

    # Define an adaptive weighted rejection sampler to sample tokens from the constrained model.
    sampler = AWRS(LLM, potential)

    # Run SMC to sample sequences from the constrained model.
    sequences = await sampler.smc(
        n_particles=5,
        ess_threshold=0.5,
        max_tokens=100,
    )

    # Return the sampled sequences and their probabilities as a ModelOutput.
    return ModelOutput(
        responses=[
            ModelResponse(response=sequence, weight=prob)
            for sequence, prob in sequences.decoded_posterior.items()
        ],
    )

### Run the evaluation

In [ ]:
from genlm.eval import run_evaluation

results = await run_evaluation(
    dataset=dataset,
    model=model,
    evaluator=evaluator,
    max_instances=2,
    n_replicates=1,
    verbosity=1,
    # output_dir="goal_inference_results", optionally save the results to a directory
)

## References

Max Zuo, Francisco Piedrahita Velez, Xiaochen Li, Michael L. Littman, and Stephen H. Bach. Planetarium: a rigorous benchmark for translating text to structured planning languages. arXiv preprint arXiv:2407.03321, 2024. URL https://arxiv.org/abs/2407.03321